# Token Prediction Dynamics

How predictions evolve through the model: which token is predicted
at each layer, confidence changes, rank tracking, and stability.

## Why This Matters

Token prediction dynamics tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_prediction_dynamics import (
    prediction_trajectory, prediction_confidence_evolution,
    prediction_rank_tracking, prediction_stability,
    token_prediction_dynamics_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Prediction Trajectory

Which token does the model predict at each layer?

In [ ]:
result = prediction_trajectory(model, tokens, position=-1)
print(f"Total prediction changes: {result['n_changes']}")
for p in result['per_layer']:
    flag = ' [CHANGED]' if p['changed_from_previous'] else ''
    print(f"  Layer {p['layer']}: token={p['predicted_token']}, "
          f"prob={p['predicted_prob']:.4f}{flag}")

## Confidence Evolution

How does entropy and max probability change through layers?

In [ ]:
result = prediction_confidence_evolution(model, tokens, position=-1)
print(f"Final entropy: {result['final_entropy']:.4f}")
for p in result['per_layer']:
    bar = '█' * int(p['max_prob'] * 40)
    print(f"  Layer {p['layer']}: H={p['entropy']:.4f}, "
          f"max_p={p['max_prob']:.4f} {bar}")

## Rank Tracking

Track how specific tokens' ranks evolve through the model.

In [ ]:
result = prediction_rank_tracking(model, tokens, position=-1, track_tokens=[1, 15, 30])
print(f"Tracking tokens: {result['tracked_tokens']}")
for p in result['per_layer']:
    ranks_str = ', '.join(f'tok{t}=#{r}' for t, r in p['ranks'].items())
    print(f"  Layer {p['layer']}: {ranks_str}")

## Prediction Stability

Cosine similarity of logit vectors between consecutive layers.

In [ ]:
result = prediction_stability(model, tokens, position=-1)
print(f"Mean stability: {result['mean_stability']:.4f}")
for p in result['per_transition']:
    print(f"  L{p['from_layer']}->L{p['to_layer']}: cos={p['cosine_similarity']:.4f}")

## Summary

In [ ]:
result = token_prediction_dynamics_summary(model, tokens, position=-1)
print(f"Changes: {result['n_changes']}, entropy: {result['final_entropy']:.4f}, "
      f"stability: {result['mean_stability']:.4f}")